# Microstructure Feature Engineering

This notebook constructs interpretable limit order book features from FI-2010 data.

The previous notebooks used the 144 benchmark features directly. Here, we focus on economically meaningful order book quantities such as mid-price, spread, depth, queue imbalance, and microprice.

We use the DecPre version of FI-2010 because it preserves price and volume information in a more interpretable scale than the z-score normalized version.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from lob_forecasting.data import load_fi2010_split
from lob_forecasting.features import (
    add_basic_lob_features,
    add_depth_features,
    add_rolling_context_features,
    make_lob_column_names,
    make_lob_dataframe,
)


In [3]:
X_train_decpre, y_train_all, X_test_decpre, y_test_all = load_fi2010_split(
    project_root=PROJECT_ROOT,
    market="NoAuction",
    normalization="DecPre",
    cf=1,
    verbose=True
)

X_train: (39512, 144)
y_train_all: (39512, 5)
X_test: (38397, 144)
y_test_all: (38397, 5)


In [ ]:
lob_columns = make_lob_column_names(n_levels=10)

lob_train = make_lob_dataframe(X_train_decpre, n_levels=10)
lob_test = make_lob_dataframe(X_test_decpre, n_levels=10)

lob_train.head()


,ask_price_1,ask_size_1,bid_price_1,bid_size_1,ask_price_2,ask_size_2,bid_price_2,bid_size_2,ask_price_3,ask_size_3,...,bid_price_8,bid_size_8,ask_price_9,ask_size_9,bid_price_9,bid_size_9,ask_price_10,ask_size_10,bid_price_10,bid_size_10
0,0.2615,0.00353,0.2606,0.00326,0.2618,0.00200,0.2604,0.00682,0.2619,0.00164,...,0.2591,0.00134,0.2629,0.00146,0.2588,0.00123,0.2633,0.00311,0.2579,0.00128
1,0.2615,0.00211,0.2606,0.00326,0.2619,0.00164,0.2604,0.00682,0.2620,0.00138,...,0.2593,0.00143,0.2637,0.00165,0.2591,0.00134,0.2646,0.00138,0.2588,0.00123
2,0.2614,0.00122,0.2606,0.00326,0.2615,0.00200,0.2604,0.00682,0.2617,0.00361,...,0.2593,0.00143,0.2629,0.00146,0.2591,0.00134,0.2633,0.00311,0.2588,0.00123
3,0.2614,0.00322,0.2606,0.00326,0.2617,0.00938,0.2604,0.00682,0.2619,0.00850,...,0.2591,0.00134,0.2637,0.00165,0.2588,0.00123,0.2646,0.00138,0.2579,0.00128
4,0.2614,0.00322,0.2606,0.00326,0.2617,0.00938,0.2604,0.00682,0.2619,0.00850,...,0.2591,0.00134,0.2637,0.00165,0.2588,0.00123,0.2646,0.00138,0.2579,0.00128


## Basic top-of-book features

We first construct features from the best bid and best ask. These are the most important prices in the limit order book because they define the current executable quote.

In [12]:
basic_train_features = add_basic_lob_features(lob_train)
basic_test_features = add_basic_lob_features(lob_test)


In [14]:
basic_train_features.head()

,mid_price,spread,level_1_imbalance,microprice,microprice_deviation
0,0.26105,0.0009,-0.039764,0.261032,-0.000018
1,0.26105,0.0009,0.214153,0.261146,0.000096
2,0.26100,0.0008,0.455357,0.261182,0.000182
3,0.26100,0.0008,0.006173,0.261002,0.000002
4,0.26100,0.0008,0.006173,0.261002,0.000002


In [16]:
train_features = add_basic_lob_features(lob_train)
train_features = add_depth_features(lob_train, train_features)

test_features = add_basic_lob_features(lob_test)
test_features = add_depth_features(lob_test, test_features)

train_features.head()

,mid_price,spread,level_1_imbalance,microprice,microprice_deviation,bid_depth_5,ask_depth_5,depth_imbalance_5,bid_depth_10,ask_depth_10,depth_imbalance_10
0,0.26105,0.0009,-0.039764,0.261032,-0.000018,0.02846,0.01400,0.340556,0.03474,0.03631,-0.022097
1,0.26105,0.0009,0.214153,0.261146,0.000096,0.02687,0.01208,0.379718,0.03346,0.02755,0.096869
2,0.26100,0.0008,0.455357,0.261182,0.000182,0.02687,0.00884,0.504901,0.03346,0.02823,0.084779
3,0.26100,0.0008,0.006173,0.261002,0.000002,0.02846,0.02793,0.009399,0.03474,0.03703,-0.031907
4,0.26100,0.0008,0.006173,0.261002,0.000002,0.02846,0.02793,0.009399,0.03474,0.03703,-0.031907


## Predictive test using custom microstructure features

We first test whether the engineered microstructure features can predict horizon-3 mid-price movement. This gives a direct comparison against the earlier 144-feature benchmark models.

In [20]:
target_col = 3

y_train = y_train_all[:, target_col].astype(int)
y_test = y_test_all[:, target_col].astype(int)

print("X custom train:", train_features.shape)
print("X custom test:", test_features.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X custom train: (39512, 11)
X custom test: (38397, 11)
y_train: (39512,)
y_test: (38397,)


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

custom_logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
)

custom_logreg.fit(train_features, y_train)

y_pred_custom_logreg = custom_logreg.predict(test_features)

print("Custom microstructure Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_custom_logreg))
print("Macro F1:", f1_score(y_test, y_pred_custom_logreg, average="macro"))
print(classification_report(y_test, y_pred_custom_logreg))

Custom microstructure Logistic Regression
Accuracy: 0.3450269552308774
Macro F1: 0.3355681970418701
              precision    recall  f1-score   support

           1       0.38      0.34      0.36     13764
           2       0.29      0.22      0.25     11749
           3       0.35      0.47      0.40     12884

    accuracy                           0.35     38397
   macro avg       0.34      0.34      0.34     38397
weighted avg       0.34      0.35      0.34     38397



In [23]:
from sklearn.ensemble import HistGradientBoostingClassifier

custom_hgb = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

custom_hgb.fit(train_features, y_train)
y_pred_custom_hgb = custom_hgb.predict(test_features)

print("Custom microstructure HistGradientBoosting")
print("Accuracy:", accuracy_score(y_test, y_pred_custom_hgb))
print("Macro F1:", f1_score(y_test, y_pred_custom_hgb, average="macro"))
print(classification_report(y_test, y_pred_custom_hgb))

Custom microstructure HistGradientBoosting
Accuracy: 0.36831002422064224
Macro F1: 0.336787370145555
              precision    recall  f1-score   support

           1       0.38      0.55      0.45     13764
           2       0.33      0.13      0.18     11749
           3       0.36      0.40      0.38     12884

    accuracy                           0.37     38397
   macro avg       0.36      0.36      0.34     38397
weighted avg       0.36      0.37      0.34     38397



In [24]:
custom_results = pd.DataFrame([
    {
        "feature_set": "custom_microstructure",
        "model": "Logistic regression",
        "horizon": 3,
        "accuracy": accuracy_score(y_test, y_pred_custom_logreg),
        "macro_f1": f1_score(y_test, y_pred_custom_logreg, average="macro"),
    },
    {
        "feature_set": "custom_microstructure",
        "model": "HistGradientBoosting",
        "horizon": 3,
        "accuracy": accuracy_score(y_test, y_pred_custom_hgb),
        "macro_f1": f1_score(y_test, y_pred_custom_hgb, average="macro"),
    },
])

custom_results

,feature_set,model,horizon,accuracy,macro_f1
0,custom_microstructure,Logistic regression,3,0.345027,0.335568
1,custom_microstructure,HistGradientBoosting,3,0.368310,0.336787


In [25]:
RESULTS_DIR = PROJECT_ROOT / "experiments"
RESULTS_DIR.mkdir(exist_ok=True)

output_path = RESULTS_DIR / "custom_microstructure_results_horizon3.csv"
custom_results.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: /Users/sohamaggarwal/Desktop/limit-order-book-forecasting/experiments/custom_microstructure_results_horizon3.csv


## Observation

The handcrafted microstructure features improve substantially over the majority-class baseline but underperform the full 144-feature FI-2010 benchmark representation.

For horizon 3, custom-feature logistic regression and HistGradientBoosting both achieve macro F1 around 0.336, compared with approximately 0.431 for logistic regression and 0.508 for HistGradientBoosting using the full 144 benchmark features.

This suggests that spread, depth, imbalance, and microprice contain meaningful but incomplete predictive information. The engineered features are especially weak on the stationary/flat class, indicating that top-of-book and depth-pressure signals are better at capturing directional pressure than no-move regimes.

In [30]:
train_features_context = add_rolling_context_features(train_features)
test_features_context = add_rolling_context_features(test_features)
train_features_context = train_features_context.fillna(0)
test_features_context = test_features_context.fillna(0)

train_features_context.head()


,mid_price,spread,level_1_imbalance,microprice,microprice_deviation,bid_depth_5,ask_depth_5,depth_imbalance_5,bid_depth_10,ask_depth_10,...,mid_return_5,mid_return_10,rolling_vol_10,rolling_vol_50,spread_mean_10,spread_std_10,imbalance_mean_10,imbalance_std_10,microprice_dev_mean_10,microprice_dev_std_10
0,0.26105,0.0009,-0.039764,0.261032,-0.000018,0.02846,0.01400,0.340556,0.03474,0.03631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.26105,0.0009,0.214153,0.261146,0.000096,0.02687,0.01208,0.379718,0.03346,0.02755,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.26100,0.0008,0.455357,0.261182,0.000182,0.02687,0.00884,0.504901,0.03346,0.02823,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.26100,0.0008,0.006173,0.261002,0.000002,0.02846,0.02793,0.009399,0.03474,0.03703,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.26100,0.0008,0.006173,0.261002,0.000002,0.02846,0.02793,0.009399,0.03474,0.03703,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
context_logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
)

context_logreg.fit(train_features_context, y_train)

y_pred_context_logreg = context_logreg.predict(test_features_context)

print("Context microstructure Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_context_logreg))
print("Macro F1:", f1_score(y_test, y_pred_context_logreg, average="macro"))
print(classification_report(y_test, y_pred_context_logreg))

Context microstructure Logistic Regression
Accuracy: 0.6285386879183269
Macro F1: 0.6286158218922101
              precision    recall  f1-score   support

           1       0.66      0.59      0.62     13764
           2       0.62      0.65      0.63     11749
           3       0.61      0.65      0.63     12884

    accuracy                           0.63     38397
   macro avg       0.63      0.63      0.63     38397
weighted avg       0.63      0.63      0.63     38397



In [32]:
context_hgb = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

context_hgb.fit(train_features_context, y_train)

y_pred_context_hgb = context_hgb.predict(test_features_context)

print("Context microstructure HistGradientBoosting")
print("Accuracy:", accuracy_score(y_test, y_pred_context_hgb))
print("Macro F1:", f1_score(y_test, y_pred_context_hgb, average="macro"))
print(classification_report(y_test, y_pred_context_hgb))

Context microstructure HistGradientBoosting
Accuracy: 0.800218767091179
Macro F1: 0.7995993056556675
              precision    recall  f1-score   support

           1       0.84      0.80      0.82     13764
           2       0.74      0.82      0.78     11749
           3       0.82      0.78      0.80     12884

    accuracy                           0.80     38397
   macro avg       0.80      0.80      0.80     38397
weighted avg       0.80      0.80      0.80     38397



## Observations

The first custom microstructure feature set used only current-snapshot order book quantities: mid-price, spread, level-1 imbalance, microprice deviation, and multi-level depth imbalance. These features contained some predictive signal, but underperformed the full 144-feature FI-2010 benchmark representation. In particular, the basic custom feature models struggled with the stationary/flat class, suggesting that one-snapshot pressure variables alone are not enough to identify no-move regimes.

Adding recent-history context changed the result substantially. Features such as recent mid-price returns, rolling volatility, rolling spread statistics, rolling imbalance, and rolling microprice-deviation statistics produced a large improvement in both accuracy and macro F1. This suggests that short-horizon FI-2010 labels are strongly related not only to the current order book state, but also to recent mid-price dynamics and persistent microstructure pressure.

In [36]:
microstructure_results = pd.DataFrame([
    {
        "feature_set": "basic_microstructure",
        "model": "Logistic regression",
        "horizon": 3,
        "accuracy": accuracy_score(y_test, y_pred_custom_logreg),
        "macro_f1": f1_score(y_test, y_pred_custom_logreg, average="macro"),
    },
    {
        "feature_set": "basic_microstructure",
        "model": "HistGradientBoosting",
        "horizon": 3,
        "accuracy": accuracy_score(y_test, y_pred_custom_hgb),
        "macro_f1": f1_score(y_test, y_pred_custom_hgb, average="macro"),
    },
    {
        "feature_set": "context_microstructure",
        "model": "Logistic regression",
        "horizon": 3,
        "accuracy": accuracy_score(y_test, y_pred_context_logreg),
        "macro_f1": f1_score(y_test, y_pred_context_logreg, average="macro"),
    },
    {
        "feature_set": "context_microstructure",
        "model": "HistGradientBoosting",
        "horizon": 3,
        "accuracy": accuracy_score(y_test, y_pred_context_hgb),
        "macro_f1": f1_score(y_test, y_pred_context_hgb, average="macro"),
    },
])

microstructure_results

,feature_set,model,horizon,accuracy,macro_f1
0,basic_microstructure,Logistic regression,3,0.345027,0.335568
1,basic_microstructure,HistGradientBoosting,3,0.368310,0.336787
2,context_microstructure,Logistic regression,3,0.628539,0.628616
3,context_microstructure,HistGradientBoosting,3,0.800219,0.799599


In [37]:
RESULTS_DIR = PROJECT_ROOT / "experiments"
RESULTS_DIR.mkdir(exist_ok=True)

output_path = RESULTS_DIR / "microstructure_feature_results_horizon3.csv"
microstructure_results.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: /Users/sohamaggarwal/Desktop/limit-order-book-forecasting/experiments/microstructure_feature_results_horizon3.csv
